# [0] Naive Trial with TTT Layer + Gemma3 + RoBERTa FC

:reference: https://github.com/test-time-training/ttt-lm-pytorch

:suggesting paper: https://arxiv.org/abs/2407.04620

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import BalancedSWUnivDaconDataset

from transformers import Gemma3Model, RobertaForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from torch.utils.data import DataLoader
from torch.nn import functional as F
from torch import nn
import torch

from lattent import TTTPreTrainedModel, TTTConfig, Block as TTTBlock, RMSNorm as TTTRMSNorm

import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import sys
import gc

In [ ]:
for test in tqdm(range(1000), desc="Testing tqdm"):
    pass

### Check GPU Availability

In [ ]:
!nvidia-smi

In [ ]:
# Set CUDA Device
device_num = 0

if torch.cuda.is_available() and device_num != -1:
    torch.cuda.set_device(device_num)
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    device_num = -1  # cpu
print(f"INFO: Using device - {device}:{device_num}")

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

In [ ]:
base_model_id = "google/gemma-3-4b-pt"
classifier_model_id = "openai-community/roberta-large-openai-detector"

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
from typing import Optional

class TTTForNaiveTextDetection(TTTPreTrainedModel):
    def __init__(
        self,
        config: TTTConfig = TTTConfig(num_hidden_layers=6),
        quantization_config: Optional[BitsAndBytesConfig] = None
    ):
        base = Gemma3Model.from_pretrained(
            base_model_id,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            quantization_config=quantization_config
        )
        self.padding_idx = base.pad_token_id
        self.vocab_size = base.vocab_size

        config.vocab_size = self.vocab_size  # Gemma3 vocab size
        config.hidden_size = 2560  # Gemma3 hidden size
        config.intermediate_size = config.hidden_size * 2
        config.class_feature_size = 1024  # RoBERTa large feature size
        super().__init__(config)

        # 0. Register base model - Gemma3Model
        self.base = base
        for param in self.base.parameters():
            param.requires_grad = False  # Freeze the base model parameters

        # 2. Main TTT model
        self.model = nn.ModuleList([TTTBlock(config, layer_idx) for layer_idx in range(config.num_hidden_layers)])
        self.bridge = nn.Linear(config.hidden_size, config.class_feature_size)
        self.norm = TTTRMSNorm(config.class_feature_size, eps=config.rms_norm_eps)
        self.gradient_checkpointing = False

        # 3. Final classification layer from RoBERTa
        self.classifier = RobertaForSequenceClassification.from_pretrained(classifier_model_id).classifier
        for param in self.classifier.parameters():
            param.requires_grad = False  # Freeze the classifier model parameters

        # 4. Initialize weights and apply final processing
        self.post_init()

    def forward(
        self,
        input_ids: torch.LongTensor = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
    ) -> torch.Tensor:
        with torch.no_grad():
            if inputs_embeds is None:
                inputs_embeds = self.base.get_input_embeddings()(input_ids)

            if attention_mask is None:
                attention_mask = torch.ones_like(input_ids)

            if position_ids is None:
                seqlen_offset = 0
                position_ids = torch.arange(
                    seqlen_offset, seqlen_offset + inputs_embeds.shape[1],
                    dtype=torch.long, device=inputs_embeds.device,
                ).unsqueeze(0)

            hidden_states = self.base.language_model(
                attention_mask=attention_mask,
                position_ids=position_ids,
                inputs_embeds=inputs_embeds,
                return_dict=True
            ).last_hidden_state

        for decoder_layer in self.model:
            if self.gradient_checkpointing and self.training:
                hidden_states = self._gradient_checkpointing_func(
                    decoder_layer.__call__,
                    hidden_states,
                    attention_mask,
                    position_ids,
                )
            else:
                hidden_states = decoder_layer(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                )

        hidden_states = self.norm(self.bridge(hidden_states[:, -1:, :]))
        return self.classifier(hidden_states)

In [ ]:
model = TTTForNaiveTextDetection()
model.to(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(
        batch, return_tensors="pt", return_token_type_ids=False,
        padding="longest" if len(batch) > 1 else False
    ).to(device)

In [ ]:
tokenize("Hello, my dog is cute")

## Train and Evaluate

In [ ]:
eval_steps = 100

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500)

In [ ]:
BATCH_SIZE = 1, 1, 1

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE[0], shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE[1], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE[2], shuffle=False, collate_fn=lambda x: x)

In [ ]:
def to_label(scores):
    return torch.argmax(scores, dim=-1)

def accuracy(scores, preds, labels):
    correct, true_human, false_human = 0, 0, 0
    for score, pred, label in zip(scores, preds, labels):
        if pred == label:
            if label == 0: true_human += 1
            correct += 1
            print(f"INFO: Correct prediction - expected {label}, got {pred} {score.tolist()}")
        else:
            if label == 0: false_human += 1
            print(f"ERROR: Incorrect prediction - expected {label}, got {pred} {score.tolist()}")
    return correct, true_human, false_human

In [ ]:
def valid_sample(sample_amount=5):
    while True:
        human_count, ai_count = 0, 0
        for texts, labels in DataLoader(valid_dataset, batch_size=1, shuffle=True):
            if labels[0] == 0:
                if human_count >= sample_amount: continue
                human_count += 1
            else:
                if ai_count >= sample_amount: continue
                ai_count += 1

            yield texts, labels

In [ ]:
with tqdm(train_loader, desc="[Training]") as progress:
    corrects, errors, true_human, false_human = 0, 0, 0, 0
    for step, (texts, labels) in enumerate(progress):
        model.train()
        try:
            optimizer.zero_grad()

            logits = model(**tokenize(texts))
            loss = criterion(logits, labels.to(device))
            loss.backward()

            optimizer.step()
            scheduler.step()
        except Exception as e:
            print(e, file=sys.stderr)

        if step % eval_steps == 0:
            model.eval()
            for texts, labels in valid_sample(25):
                try:
                    torch.cuda.empty_cache()
                    gc.collect()
                    with torch.no_grad():
                        logits = model(**tokenize(texts))
                        scores = torch.softmax(logits, dim=-1)
                        preds = to_label(scores)

                        c, th, fh = accuracy(scores, preds, labels.tolist())
                        corrects += c
                        errors += len(labels) - c
                        true_human += th
                        false_human += fh
                except Exception as e:
                    print(e, file=sys.stderr)

                progress.set_description(f"[Validating] Correct: {corrects/(corrects+errors):.6%} [H: {true_human}, A: {corrects-true_human}], Errors: {errors} [H: {false_human}, A: {errors-false_human}]")
            progress.set_description(f"[Training] Correct: {corrects/(corrects+errors):.6%} [H: {true_human}, A: {corrects-true_human}], Errors: {errors} [H: {false_human}, A: {errors-false_human}]")

In [ ]:
with tqdm(valid_loader, desc="[Validating]") as progress:
    corrects, errors, true_human, false_human = 0, 0, 0, 0
    model.eval()
    for texts, labels in progress:
        torch.cuda.empty_cache()
        gc.collect()

        try:
            with torch.no_grad():
                logits = model(**tokenize(texts))
                scores = torch.softmax(logits, dim=-1)
                preds = to_label(scores)

                c, th, fh = accuracy(scores, preds, labels.tolist())
                corrects += c
                errors += len(labels) - c
                true_human += th
                false_human += fh
        except Exception as e:
            if "CUDA" in str(e):
                print(f"ERROR: {e} - {texts}")
            else:
                raise e

        progress.set_description(f"[Validating] Correct: {corrects/(corrects+errors):.6%} [H: {true_human}, A: {corrects-true_human}], Errors: {errors} [H: {false_human}, A: {errors-false_human}]")

In [ ]:
results, possibilities = [], []
with tqdm(test_loader, desc="[Testing]") as progress:
    humans, ais = 0, 0
    model.eval()
    for datas in progress:
        texts, _ = zip(*datas)
        torch.cuda.empty_cache()
        gc.collect()

        try:
            with torch.no_grad():
                logits = model(**tokenize(texts))
                scores = torch.softmax(logits, dim=-1)
                preds = to_label(scores)
                possibilities.extend(scores.cpu().tolist())
                results.extend(preds.cpu().tolist())
                counter = Counter(preds.cpu().tolist())
                humans += counter[0]
                ais += counter[1]
        except Exception as e:
            if "CUDA" in str(e):
                print(f"ERROR: {e} - {texts}")
            else:
                raise e

        counter = Counter(results)
        progress.set_description(f"[Testing] Human: {counter[0]/len(test_dataset):.2%}, Ai: {counter[1]/len(test_dataset):.2%}")

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')
sub

In [ ]:
sub['generated'] = results
sub

In [ ]:
plt.figure(figsize=(12, 7))
sns.histplot(data=sub, x='generated', kde=True, bins=50)
plt.title("Prediction Probability Distribution", fontsize=16)
plt.xlabel("Predicted Probability (Generated = 1)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)

plt.show()
print(sub['generated'].describe())

In [ ]:
sub.to_csv("./data/submission_naive_ttt.csv", index=False, encoding='utf-8-sig')